# 008 - Baseline comparison

Phase 7 told me the fine-tuned checkpoint works, but it didn't tell me how much of that came from fine-tuning at all. BiomedCLIP was already trained on a large set of biomedical image-text pairs before I touched it, so some of what I measured in phase 7 could just be BiomedCLIP already being good at biomedical images in general, not anything specific to my fine-tuning.

The fix is a proper baseline: run the exact same evaluation (Recall@K, k-means clustering) on the pretrained BiomedCLIP checkpoint, with zero fine-tuning, on the same held out test split. Whatever gap I see between zero-shot and fine-tuned is the actual, isolated contribution of everything I did in phase 6.

## What zero-shot means here

Zero-shot just means loading BiomedCLIP straight from its pretrained weights and using it exactly as-is, no gradient updates, no exposure to my dataset at all. It has never seen a RadiologyNET image or my translated diagnosis text. Whatever it can already retrieve correctly, it's doing purely off what it learned during its own original pretraining.

I already had `load_biomedclip()` in src/models/biomedclip.py from phase 5, so this only needed one small addition to src/evaluation/embed.py, a `load_zeroshot_model` function that loads the pretrained model straight to eval mode with no checkpoint loading step, sitting right next to the existing `load_trained_model` which does the same thing but then overwrites the weights with checkpoints/best.pt.

Everything downstream, the dataset, the preprocessing, the Recall@K code, the clustering code, is exactly the same code from phase 7. Only the model weights differ. That's what makes this a clean comparison instead of a different experiment.

## Computing zero-shot embeddings

Ran `get_split_embeddings('test', model, preprocess_val, tokenizer, cache_name='zeroshot')` with the zero-shot model, same as phase 7's `cache_name='finetuned'` call. It caches to data/embeddings/zeroshot_test.pt so I'm not recomputing this every time.

Then the same functions from phase 7 on top of it, unmodified:
- `text_to_image_recall` / `image_to_text_recall` from src/evaluation/retrieval.py
- `cluster_embeddings` / `evaluate_against_modality` from src/evaluation/clustering.py

Results saved to results/phase8_metrics.json, same structure as phase7_metrics.json so the two are directly diffable.

## Recall@K, zero-shot versus fine-tuned

Text to image (999 unique diagnosis queries, pool of 3046 images):

| K | zero-shot | fine-tuned | gain |
|---|---|---|---|
| 1 | 8.2% | 9.7% | +1.5 pts |
| 5 | 22.5% | 29.2% | +6.7 pts |
| 10 | 33.7% | 43.7% | +10.0 pts |

Image to text (3046 image queries, pool of 999 unique diagnoses):

| K | zero-shot | fine-tuned | gain |
|---|---|---|---|
| 1 | 6.1% | 10.1% | +4.0 pts |
| 5 | 22.9% | 36.5% | +13.6 pts |
| 10 | 37.1% | 56.1% | +19.0 pts |

Two things stand out. First, zero-shot is not bad. Recall@10 of 33.7% and 37.1% against random-chance baselines of 0.3% and 1.0% means the pretrained model already has real, transferable signal for this task before I did anything. That makes sense given BiomedCLIP was trained on a broad set of biomedical image-text pairs, radiological images and their descriptions are exactly the kind of thing it should already have some grip on.

Second, fine-tuning clearly helps, and it helps more in the image to text direction than text to image (+19 points at K=10 versus +10 points). My best guess for the asymmetry is that the text side is the more unusual part of my setup relative to what BiomedCLIP originally saw, my diagnosis text is machine-translated from Croatian and comes from one specific institution's phrasing conventions, so there's more room for fine-tuning to adapt the text tower to this particular distribution of language than there is for the image tower to adapt to fairly standard-looking radiological images.

## Clustering, zero-shot versus fine-tuned

K-means with k=5, scored against the real Modality label via ARI and NMI:

| | zero-shot ARI | zero-shot NMI | fine-tuned ARI | fine-tuned NMI |
|---|---|---|---|---|
| image embeds | 0.729 | 0.781 | 0.945 | 0.928 |
| text embeds | 0.699 | 0.818 | 0.860 | 0.890 |

(fine-tuned numbers are the cached ones from phase 7, the ones saved in results/phase7_metrics.json. Phase 7 also noted a run to run variation in the image ARI specifically, 0.769 on the first measurement versus 0.945 on the cached one, caused by bf16 numerical noise landing differently on the RF/XA boundary. That caveat still applies here, so I'm reading the fine-tuned image ARI as somewhere in the high 0.7s to mid 0.9s range rather than pinning it to one decimal.)

Even zero-shot recovers modality far better than chance (ARI 0 would mean no better than random). That's a real finding on its own, BiomedCLIP's pretrained embedding space already separates radiological modalities reasonably well without ever seeing this dataset, which lines up with the Recall@K result above.

What's more interesting is where zero-shot gets confused, because it's not the same place fine-tuned gets confused. Looking at the actual crosstabs:

Zero-shot image embeds: CT and MR each land in their own clean cluster (551/553 and 468/489). RF never gets a clean cluster of its own, all 320 RF images end up folded into a mixed cluster together with 229 XA images. This is the same RF/XA confusion phase 7 already flagged for the fine-tuned model, so it's not something fine-tuning introduced, it's already there zero-shot and fine-tuning only partially resolves it.

Zero-shot text embeds: the confusion is in a completely different place. CT and MR get merged into one shared cluster (567 CT and 469 MR together), while RF actually gets its own clean cluster this time (320/328). So the zero-shot text tower confuses CT with MR, and the zero-shot image tower confuses RF with XA, two different failure modes on two different modality pairs. My reading is that this reflects what's actually hard about each modality: CT and MR are both cross-sectional imaging and can get described in overlapping language (slices, sequences, contrast) even though they look visually distinct, while RF and XA are both dynamic fluoroscopy-style contrast procedures that can look visually similar even when the accompanying text is clearly different. Fine-tuning cleans up both of these to varying degrees, but the underlying difficulty per modality pair was already baked into BiomedCLIP's pretraining, not something I introduced or fully solved.

## Correction from phase 9: the clustering conclusion above is wrong

The section above concludes that fine-tuning improves clustering purity on both embedding types. For image embeddings that is not true, and the reason is a bug rather than a judgement call.

Every clustering number above, for both zero-shot and fine-tuned, was computed while RadiologyNETDataset was still picking `random.choice(slices)` on the test split. Images with more than one slice, roughly 5 percent of rows and almost all XA, got a different picture on every recomputation. So those two ARI figures were not two measurements of the same thing, and phase 7 had already seen the same checkpoint swing between 0.7692 and 0.9453 on separate runs.

Recomputed in phase 9 with the slice choice made deterministic, both models through the identical pipeline, test split:

| model | image ARI | text ARI |
|---|---|---|
| zero-shot BiomedCLIP | 0.7188 | 0.6987 |
| phase 6/7 fine-tuned checkpoint | 0.6660 | 0.8597 |

So on image embeddings zero-shot is actually the better of the two, 0.7188 against 0.6660. Fine-tuning slightly degrades how cleanly modality separates in image space rather than improving it. The 0.9453 figure this section relied on was a lucky draw from the random slice bug, not a real result.

The text side survives, and survives strongly: 0.8597 fine-tuned against 0.6987 zero-shot. That one is trustworthy for a specific reason, diagnosis text does not depend on which slice was picked, so the text embeddings were never affected by the bug. Phase 9 confirmed this directly, the recomputed fine-tuned text ARI came back bit for bit identical to the number recorded here.

Making sense of the corrected picture: nothing in the training objective asks for modality separation. The contrastive loss aligns an image with its diagnosis text, and modality structure in the image embeddings is incidental structure inherited from BiomedCLIP's pretraining, so there is no particular reason fine-tuning would preserve it and some reason it would blur it. The text side improves because the diagnosis text is exactly what the model is being fitted to, and Croatian radiology reports describe different modalities in systematically different language.

What still stands from this phase: the retrieval comparison, which was recomputed in phase 9 and holds up (zero-shot Recall@10 of 0.3333 and 0.3723, fine-tuned 0.4414 and 0.5611), and the finding that zero-shot BiomedCLIP is already far above chance before any fine-tuning. What does not stand is the claim that fine-tuning improves clustering across the board. It improves the text side and mildly hurts the image side.

The observation further up that zero-shot image and text embeddings confuse different modality pairs also needs treating with caution, since it was read off crosstabs computed under the same bug.

## Summary and what's next

Fine-tuning is doing real, measurable work, not just marginal polish. Recall@10 improved by 10 to 19 points over an already-far-above-chance zero-shot baseline, and clustering purity improved on both embedding types, most visibly on the RF/XA boundary in the image embeds. At the same time, the zero-shot baseline being this strong to start with says a good chunk of the ceiling here was always going to come from BiomedCLIP's original pretraining, not from my ~10,000 exams. That's a reasonable thing to say plainly in the thesis rather than presenting the fine-tuned numbers as if they came from nothing.

Next is phase 9, pulling together the final plots and tables for the thesis write-up itself, across all phases, not just this one.